In [ ]:
# !pip install transformers

In [ ]:
from transformers import pipeline

In [ ]:
# ① 감성 분석 (영어)
sentiment = pipeline("sentiment-analysis")
result = sentiment("I love this course!")
print(result)  # [{'label': 'POSITIVE', 'score': 0.9998}]

In [ ]:
# ② 텍스트 분류 (한국어 - 사전학습된 모델 사용)
classifier = pipeline(
    "text-classification",
    model="snunlp/KR-FinBert-SC"  # 한국어 금융 감성분석
)
result = classifier("이 주식은 앞으로 크게 오를 것 같습니다.")
print(result)  # [{'label': 'positive', 'score': 0.92}]

In [ ]:
# ③ 개체명 인식(NER)
ner = pipeline("ner", model="Leo97/KoELECTRA-small-v3-modu-ner")
result = ner("삼성전자는 서울에 본사를 두고 있다.")
for entity in result:
    print(f"{entity['word']} → {entity['entity']} ({entity['score']:.2f})")
# 삼성전자 → B-ORG (0.98)
# 서울 → B-LOC (0.99)

In [ ]:
# ④ 마스크 채우기 (MLM 시연)
fill_mask = pipeline("fill-mask", model="klue/bert-base")
result = fill_mask("AI 기술이 [MASK] 발전하고 있다.")
for r in result[:3]:
    print(f"{r['token_str']}: {r['score']:.3f}")
# 빠르게: 0.312
# 급속도로: 0.198
# 지속적으로: 0.145

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

text = "딥러닝 모델 훈련 방법"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

# last_hidden_state: 각 토큰의 문맥화된 표현
print(outputs.last_hidden_state.shape)  # [1, seq_len, 768]

# [CLS] 토큰 벡터 = 문장 전체 표현 (분류 작업에 활용)
cls_embedding = outputs.last_hidden_state[:, 0, :]
print(cls_embedding.shape)  # [1, 768]

# pooler_output: [CLS]를 Linear+Tanh 통과한 결과
print(outputs.pooler_output.shape)  # [1, 768]
